# Phase A — Learn the machinery on tiny data - Warming up :)

**WIR 2026 · TH Köln** — based on the official `pyterrier-intro`
tutorial ([irgroup-classrooms/wir-2026](https://github.com/irgroup-classrooms/wir-2026)).

Before we touch the 870k LongEval-Sci papers, we learn how PyTerrier works on a
*tiny* collection (`data/ai.json`). By the end we
will understand:

1. how an **inverted index** (the lexicon / term-document matrix) is built and read,
2. what the **TF** and **TF-IDF** weighting models actually compute, and
3. why IDF matters — the intuition we carry into the real baselines (Assignment III).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyterrier as pt

# PyTerrier runs on the JVM. Start it once (idempotent).
if not pt.java.started():
    pt.java.init()

# Resolve the repo root whether the kernel's cwd is the repo or the notebooks/ folder.
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = REPO_ROOT / "data"
print("repo root:", REPO_ROOT)

repo root: c:\Users\rafae\OneDrive\Desktop\TH_Koeln\03-Semester\Web_Information_Retrieval\WIR_Retriever_Engine


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## 1. Load & clean `ai.json`

In [2]:
with open(DATA_DIR / "ai.json", encoding="utf-8") as f:
    items = json.load(f)["items"]

df = pd.DataFrame(items)
df = df.loc[:, df.isna().mean() < 0.5]          # drop columns that are >50% empty
print(f"{len(df)} records, {df.shape[1]} non-sparse columns")
df.head()

2000 records, 11 non-sparse columns


,type,id,tags,intraHash,label,user,description,date,changeDate,count,url
0,Bookmark,https://www.bibsonomy.org/url/9a95a0ec1b661c27...,"[expert_system_prescription_support, ai, medic...",9a95a0ec1b661c277fba06ebd5044172,Artifical Intelligence Systems in Clinical Pra...,jepcastel,,2008-12-24 15:46:21,2012-11-25 18:07:27,1,http://www.openclinical.org/aisinpractice.html
1,Bookmark,https://www.bibsonomy.org/url/4faee58023ed540b...,"[OECD, artificial_intelligence]",4faee58023ed540b8a11e3471b6e1158,OECD Artificial Intelligence Papers,meneteqel,The OECD Artificial Intelligence Papers series...,2025-08-11 16:09:59,2025-08-11 16:09:59,1,https://www.oecd.org/en/publications/oecd-arti...
2,Bookmark,https://www.bibsonomy.org/url/ab1853d0737e8f05...,"[robotics, artificial_intelligence, European_C...",ab1853d0737e8f0565b737463d7d05ed,Artificial Intelligence | Shaping Europe’s dig...,meneteqel,The European Commission puts forward a Europea...,2020-02-20 10:38:30,2020-02-20 10:38:30,1,https://ec.europa.eu/digital-single-market/en/...
3,Bookmark,https://www.bibsonomy.org/url/11ed70d879ab786e...,"[ethics, machinelearning, learninganalytics, h...",11ed70d879ab786edaf38b4a41db3277,Artificial Intelligence in Higher Education – ...,ereidt,“The Promise and Peril of Artificial Intellige...,2018-08-26 11:31:16,2018-08-26 11:42:28,1,https://medium.com/@BryanFendley/artificial-in...
4,Bookmark,https://www.bibsonomy.org/url/0791349ca5df61f2...,"[compare, Studium, ai, vergleich, master, ki]",0791349ca5df61f2f740f401805d0b96,Best Masters of Science (MScs) in Artificial I...,hotho,Contact Schools Directly - Compare 27 Masters ...,2019-08-16 18:14:29,2019-08-16 18:14:29,1,https://www.masterstudies.com/MSc/Artificial-I...


## 2. Give PyTerrier the two columns it requires: `docno` and `text`

Every PyTerrier document needs a unique string id (`docno`) and the searchable
`text`. Here the title (`label`) is the text we index.

In [3]:
df["docno"] = df["id"].astype(str)
df["text"] = df["label"].fillna("")
df[["docno", "text"]].head()

,docno,text
0,https://www.bibsonomy.org/url/9a95a0ec1b661c27...,Artifical Intelligence Systems in Clinical Pra...
1,https://www.bibsonomy.org/url/4faee58023ed540b...,OECD Artificial Intelligence Papers
2,https://www.bibsonomy.org/url/ab1853d0737e8f05...,Artificial Intelligence | Shaping Europe’s dig...
3,https://www.bibsonomy.org/url/11ed70d879ab786e...,Artificial Intelligence in Higher Education – ...
4,https://www.bibsonomy.org/url/0791349ca5df61f2...,Best Masters of Science (MScs) in Artificial I...


## 3. Let's Index it

`IterDictIndexer` streams a list/iterator of dicts into a Terrier index. We pass an
**absolute** path because Terrier otherwise resolves relative paths against its own
internal `./var` home (a classic first-time gotcha).

In [4]:
index_path = REPO_ROOT / "ai_index"
index_path.mkdir(exist_ok=True)
for p in index_path.glob("*"):       # clear any stale index files from a previous run
    try:
        p.unlink()
    except OSError:
        pass

indexer = pt.IterDictIndexer(str(index_path), overwrite=True,
                             meta={"docno": 256, "text": 4096})
index_ref = indexer.index(df[["docno", "text"]].to_dict(orient="records"))
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

Number of documents: 2000
Number of terms: 2915
Number of postings: 12318
Number of fields: 0
Number of tokens: 12624
Field names: []
Positions:   false



## 4. Inspect the inverted index (the lexicon / TDM)

For each term the lexicon stores **Nt** = in how many documents the term occurs
(document frequency) and **TF** = how often it occurs in total. Let's look at the
most frequent terms — note how the top of the list is dominated by common words
(this is *exactly* why we need IDF and, later, stopword removal).

In [5]:
term_freq = {kv.getKey(): kv.getValue().getFrequency() for kv in index.getLexicon()}
top = sorted(term_freq.items(), key=lambda x: x[1], reverse=True)[:25]
pd.DataFrame(top, columns=["term", "total_frequency"])

,term,total_frequency
0,intellig,1058
1,artifici,972
2,home,535
3,page,534
4,ai,201
5,learn,118
6,system,90
7,base,87
8,us,76
9,applic,75


## 5. The first retriever — `Tf`

The raw term-frequency model scores a document by how often the query terms appear
in it, with no normalisation:

$$score(d, q) = \sum_{t \in q} tf_{t,d}$$

It is the weakest model — long documents and common words win unfairly.

In [6]:
tf = pt.terrier.Retriever(index, wmodel="Tf")
tf.search("Phishing detection nlp").head()

,qid,docid,docno,rank,score,query
0,1,491,https://www.bibsonomy.org/url/f8a1a6df86f89887...,0,2.0,Phishing detection nlp
1,1,1860,https://www.bibsonomy.org/bibtex/2dd06a15143b3...,1,2.0,Phishing detection nlp
2,1,1084,https://www.bibsonomy.org/bibtex/28580d3236777...,2,1.0,Phishing detection nlp
3,1,1114,https://www.bibsonomy.org/bibtex/2efe3507995fb...,3,1.0,Phishing detection nlp
4,1,1117,https://www.bibsonomy.org/bibtex/2376a899e3276...,4,1.0,Phishing detection nlp


## 6. Let's add IDF — the `TF_IDF` model

TF-IDF down-weights terms that appear in many documents (low information) and
rewards rare, discriminating terms:

$$score(d, q) = \sum_{t \in q} tf_{t,d}\cdot \log\frac{N}{df_t}$$

The ranking changes vs. plain `Tf` — documents that match the *rarer* query term
get promoted.

In [7]:
tfidf = pt.terrier.Retriever(index, wmodel="TF_IDF")
tfidf.search("machine learning").head()

,qid,docid,docno,rank,score,query
0,1,1420,https://www.bibsonomy.org/bibtex/2aac90bd39bb6...,0,6.590638,machine learning
1,1,163,https://www.bibsonomy.org/url/80e974da88825bdb...,1,6.062074,machine learning
2,1,1474,https://www.bibsonomy.org/bibtex/27cda56873f7e...,2,5.653492,machine learning
3,1,1121,https://www.bibsonomy.org/bibtex/21f547191fa6e...,3,5.632668,machine learning
4,1,1178,https://www.bibsonomy.org/bibtex/241ecbc00c5f2...,4,5.632668,machine learning


## 7. Demystify IDF by hand

To prove there is no magic, we recompute IDF ourselves straight from the lexicon and
collection statistics. We stem the term first because the index is stemmed.

In [8]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def idf_of(term: str) -> float:
    lex = index.getLexicon()
    stem = stemmer.stem(term.lower())
    if stem not in lex:
        return float("nan")
    df_t = lex[stem].getDocumentFrequency()
    N = index.getCollectionStatistics().getNumberOfDocuments()
    return float(np.log10(N / df_t))

for t in ["Phishing", "detection", "nlp"]:
    print(f"  idf({t:<12}) = {idf_of(t):.3f}")

  idf(Phishing    ) = 3.301
  idf(detection   ) = 1.770
  idf(nlp         ) = 3.301


> **Remarks:** We can index now, read the lexicon, and we understand TF vs.
> TF-IDF (IDF rewards rare terms; document length is still unhandled — that is what
> BM25 fixes in notebook 03). Next: **`02_inspect_and_index`** does the same on the
> real LongEval-Sci collection.